# IHC Binary Evaluation — Comparison with ElSherief et al. (2021)

This notebook replicates the binary hate speech classification evaluation from **Table 3 of "Latent Hatred: A Benchmark for Understanding Implicit Hate Speech"** (ElSherief et al., EMNLP 2021) on the IHC dataset.

**Setup:** IHC test split with explicit hate removed → binary classification (not_hate vs implicit_hate, like their experiment)  
**Models evaluated:** BERT, HateBERT, RoBERTa with both baseline (fine-tuned on raw input) and RAG (fine-tuned on retrieval-augmented inputs).

## 1. Imports

In [1]:
import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import faiss
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# rag.py file copy paste from the RAG/ folder
from rag import retrieve_top_k_above_threshold

## 2. Configuration

In [2]:
ROOT_DIR        = Path('..')   # notebook lives in evaluation/
WEIGHTS_DIR     = ROOT_DIR / 'weights'
WEIGHTS_RAG_DIR = ROOT_DIR / 'weights_rag'
INDEX_DIR       = ROOT_DIR / 'RAG' / 'index'

MAX_LENGTH = 256
BATCH_SIZE = 32

# Retrieval config (must match what was used during RAG training)
K         = 5
THRESHOLD = 0.98

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cpu


## 3. Load & Filter IHC

Same split and filter as training: 90/10 train/test (seed=42), explicit hate removed, binary labels.

In [3]:
raw_ihc = load_dataset('tasksource/implicit-hate-stg1', split='train')
splits  = raw_ihc.train_test_split(test_size=0.10, seed=42)

test_ihc_full = splits['test']
test_ihc_filtered = test_ihc_full.filter(lambda x: x['class'] != 'explicit_hate')
test_ihc = test_ihc_filtered.map(lambda x: {'label': 0 if x['class'] == 'not_hate' else 1})

print(f'IHC test — full: {len(test_ihc_full):,}  after removing explicit hate: {len(test_ihc):,}')
print(f'  Non-hate: {sum(1 for x in test_ihc if x["label"] == 0):,}')
print(f'  Implicit hate: {sum(1 for x in test_ihc if x["label"] == 1):,}')

Filter:   0%|          | 0/2148 [00:00<?, ? examples/s]

Map:   0%|          | 0/2028 [00:00<?, ? examples/s]

IHC test — full: 2,148  after removing explicit hate: 2,028
  Non-hate: 1,330
  Implicit hate: 698


## 4. Model Registry

- **Baseline** in `weights/`
- **RAG** in `weights_rag/`

In [4]:
BASELINE_MODELS = [
    {'path': WEIGHTS_DIR / 'bert-base-uncased_IHC', 'label': 'BERT (baseline)'},
    {'path': WEIGHTS_DIR / 'hateBERT_IHC',          'label': 'HateBERT (baseline)'},
    {'path': WEIGHTS_DIR / 'roberta-base_IHC',      'label': 'RoBERTa (baseline)'},
]

RAG_MODELS = [
    {
        'path':       WEIGHTS_RAG_DIR / 'bert'     / 'base' / 'IHC',
        'label':      'BERT (RAG)',
        'hf_id':      'bert-base-uncased',
        'index_key':  'bert',
        'index_type': 'training',
    },
    {
        'path':       WEIGHTS_RAG_DIR / 'hatebert' / 'base' / 'IHC',
        'label':      'HateBERT (RAG)',
        'hf_id':      'GroNLP/hateBERT',
        'index_key':  'hatebert',
        'index_type': 'training',
    },
    {
        'path':       WEIGHTS_RAG_DIR / 'roberta'  / 'base' / 'IHC',
        'label':      'RoBERTa (RAG)',
        'hf_id':      'roberta-base',
        'index_key':  'roberta',
        'index_type': 'training',
    },
]

all_models = BASELINE_MODELS + RAG_MODELS

print(f"{'Model':<25} {'Type':<10} Weights?")
print('-' * 48)
for m in all_models:
    has  = (m['path'] / 'model.safetensors').exists() or (m['path'] / 'pytorch_model.bin').exists()
    kind = 'RAG' if 'hf_id' in m else 'baseline'
    print(f"{m['label']:<25} {kind:<10} {'✓' if has else '✗  (missing)'}")

Model                     Type       Weights?
------------------------------------------------
BERT (baseline)           baseline   ✓
HateBERT (baseline)       baseline   ✓
RoBERTa (baseline)        baseline   ✓
BERT (RAG)                RAG        ✓
HateBERT (RAG)            RAG        ✓
RoBERTa (RAG)             RAG        ✓


## 5. Helpers

In [ ]:
def tokenize_plain(hf_dataset, tokenizer):
    encoded = tokenizer(
        list(hf_dataset['post']),
        truncation=True, padding='max_length', max_length=MAX_LENGTH,
    )
    encoded['labels'] = list(hf_dataset['label'])
    return Dataset.from_dict(encoded)


def augment_test(hf_dataset, ret_model, ret_tokenizer, ret_index, ret_documents):
    records = []
    for example in tqdm(hf_dataset, desc='augmenting'):
        neighbors = retrieve_top_k_above_threshold(
            example['post'], THRESHOLD, ret_model, ret_tokenizer,
            ret_index, ret_documents, chunk_id=None, k=K,
        )
        records.append({
            'query':     example['post'],
            'neighbors': [text for text, _ in neighbors],
            'label':     example['label'],
        })
    return records


def tokenize_augmented(records, tokenizer):
    sep   = tokenizer.sep_token
    texts = [f' {sep} '.join([r['query']] + r['neighbors']) for r in records]
    encoded = tokenizer(
        texts,
        truncation=True, padding='max_length', max_length=MAX_LENGTH,
    )
    encoded['labels'] = [r['label'] for r in records]
    return Dataset.from_dict(encoded)


def compute_metrics(eval_pred):
    preds  = np.argmax(eval_pred.predictions, axis=-1)
    labels = eval_pred.label_ids
    return {
        'macro_f1': f1_score(labels, preds, average='macro',  zero_division=0),
        'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
        'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
    }

: 

## 6. Evaluation Loop

In [ ]:
results = {}

eval_args = TrainingArguments(
    output_dir='./tmp_eval',
    per_device_eval_batch_size=BATCH_SIZE,
    report_to='none',
)

for entry in all_models:
    has_weights = (entry['path'] / 'model.safetensors').exists() or (entry['path'] / 'pytorch_model.bin').exists()
    if not has_weights:
        print(f"[skip] {entry['label']} — no weights on disk")
        continue

    print(f"\n{'='*55}")
    print(f"{entry['label']}")
    print(f"{'='*55}")

    is_rag = 'hf_id' in entry

    if is_rag:
        # ── Retrieval augmentation ────────────────────────────────
        print('Loading retriever and augmenting test set...')
        ret_tokenizer = AutoTokenizer.from_pretrained(entry['hf_id'])
        ret_model     = AutoModel.from_pretrained(entry['hf_id']).eval().to(device)
        ret_index     = faiss.read_index(
            str(INDEX_DIR / entry['index_key'] / 'base' / f"vdb_{entry['index_type']}.faiss")
        )
        with open(INDEX_DIR / f"lookup_{entry['index_type']}.json") as f:
            ret_documents = json.load(f)

        aug_records = augment_test(test_ihc, ret_model, ret_tokenizer, ret_index, ret_documents)

        del ret_model, ret_tokenizer
        if device.type == 'cuda':
            torch.cuda.empty_cache()

        tokenizer = AutoTokenizer.from_pretrained(entry['path'])
        tok_test  = tokenize_augmented(aug_records, tokenizer)

    else:
        # ── Plain text for baseline models ────────────────────────
        tokenizer = AutoTokenizer.from_pretrained(entry['path'])
        tok_test  = tokenize_plain(test_ihc, tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(entry['path'])

    trainer = Trainer(
        model=model,
        args=eval_args,
        compute_metrics=compute_metrics,
    )

    preds_out = trainer.predict(tok_test)
    preds  = np.argmax(preds_out.predictions, axis=-1)
    labels = list(test_ihc['label'])

    print(classification_report(labels, preds, target_names=['Non-hate', 'Implicit hate']))

    results[entry['label']] = {
        'macro_f1': f1_score(labels, preds, average='macro',  zero_division=0),
        'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
        'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
    }

    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()


BERT (baseline)


  0%|          | 0/64 [00:00<?, ?it/s]

               precision    recall  f1-score   support

     Non-hate       0.84      0.85      0.85      1330
Implicit hate       0.71      0.69      0.70       698

     accuracy                           0.80      2028
    macro avg       0.77      0.77      0.77      2028
 weighted avg       0.79      0.80      0.79      2028


HateBERT (baseline)


  0%|          | 0/64 [00:00<?, ?it/s]

               precision    recall  f1-score   support

     Non-hate       0.84      0.87      0.85      1330
Implicit hate       0.73      0.67      0.70       698

     accuracy                           0.80      2028
    macro avg       0.78      0.77      0.78      2028
 weighted avg       0.80      0.80      0.80      2028


RoBERTa (baseline)


  0%|          | 0/64 [00:00<?, ?it/s]

               precision    recall  f1-score   support

     Non-hate       0.84      0.86      0.85      1330
Implicit hate       0.73      0.69      0.71       698

     accuracy                           0.81      2028
    macro avg       0.79      0.78      0.78      2028
 weighted avg       0.80      0.81      0.81      2028


BERT (RAG)
Loading retriever and augmenting test set...


Encoding: 100%|██████████| 1/1 [00:00<00:00,  1.35it/s]

## 7. Paper Results

Here we report manually the best results of the paper from Table 3 of ElSherief et al. (2021).

In [ ]:
PAPER_RESULTS = {
    'SVM n-grams (paper)':     {'macro_f1': 64.4, 'macro_p': 61.4, 'macro_r': 67.7},
    'BERT (paper)': {'macro_f1': 68.9, 'macro_p': 72.1, 'macro_r': 66.0},
    'BERT + Aug (paper)':  {'macro_f1': 70.4, 'macro_p': 67.8, 'macro_r': 73.2},
}

## 8. Results — Comparison Table

Our models vs. published paper results.

In [ ]:
all_results = dict(results)
for label, vals in PAPER_RESULTS.items():
    if vals['macro_f1'] is not None:
        all_results[label] = vals

df = pd.DataFrame({
    label: {'Macro F1': v['macro_f1'], 'Macro Precision': v['macro_p'], 'Macro Recall': v['macro_r']}
    for label, v in all_results.items()
}).T
df.index.name = 'Model'

display(
    df.style
    .format('{:.3f}')
    .highlight_max(axis=0, props='font-weight: bold; background-color: #d4f1d4')
    .set_caption('IHC binary evaluation (implicit hate only) — ElSherief et al. (2021) comparison')
)